In [2]:
! pip install transformers torch accelerate


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_PATH = r"C:\qwen1.5b"
SYSTEM_PROMPT = "You are a helpful technical assistant. Answer clearly and accurately."
MAX_TOKENS = 512

print("Loading model... (1-2 minutes)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,   # CPU
    device_map="cpu",
)
model.eval()

print("=" * 40)
print("   Qwen2.5-1.5B  |  Offline  |  CPU")
print("   'clear' = new chat  |  'quit' = exit")
print("=" * 40 + "\n")

history = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(user_input):
    history.append({"role": "user", "content": user_input})

    prompt = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    history.append({"role": "assistant", "content": response})
    return response


while True:
    try:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "bye"):
            print("Bye!")
            break
        if user_input.lower() == "clear":
            history.clear()
            history.append({"role": "system", "content": SYSTEM_PROMPT})
            print("--- Chat cleared ---\n")
            continue
        print("Thinking...\n")
        reply = chat(user_input)
        print(f"Bot: {reply}\n")
        print("-" * 40)
    except KeyboardInterrupt:
        print("\nBye!")
        break

In [ ]:
# version 2
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = r"C:\qwen1.5b"

SYSTEM_PROMPT = """
You are an offline exam MCQ assistant.

Return:
Answer: <option letter>
Reason: <one short sentence>

Do not repeat the question.
Do not repeat all options.
Keep the response under 50 words.
"""

MAX_TOKENS = 64

# ============================================================
# LOAD MODEL
# ============================================================

print("Loading model... Please wait.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="cpu",
    local_files_only=True
)

model.eval()

print("=" * 50)
print("        OFFLINE EXAM MCQ ASSISTANT")
print("        Model : Local Qwen")
print("        Device: CPU")
print("        Precision: FP16")
print("        Max tokens: 64")
print("=" * 50)
print()
print("Type 'clear' to clear chat.")
print("Type 'quit' to exit.")
print()


# ============================================================
# CHAT FUNCTION
# ============================================================

def answer_mcq(question):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Create prompt using the model's chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # CPU inference
    with torch.no_grad():

        output = model.generate(
            **inputs,

            max_new_tokens=MAX_TOKENS,

            # Deterministic answer
            do_sample=False,

            # Avoid excessive repetition
            repetition_penalty=1.05,

            # Stop at EOS
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output[0][input_length:]

    # Decode answer
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    try:

        question = input("You: ").strip()

        if not question:
            continue

        # Exit
        if question.lower() in ["quit", "exit", "bye"]:
            print("\nBye!")
            break

        # Clear
        if question.lower() == "clear":
            print("\n--- Ready for next question ---\n")
            continue

        print("\nThinking...\n")

        answer = answer_mcq(question)

        print("Bot:")
        print(answer)

        print("\n" + "-" * 50 + "\n")

    except KeyboardInterrupt:

        print("\n\nBye!")
        break

    except Exception as e:

        print("\nError:", e)
        print()